# Heart Disease Classification Pipeline

This notebook walks through the **MLOps pipeline** for a heart disease classification project. The pipeline covers the full lifecycle:

1. Loading and cleaning raw data
2. Validating the dataset schema
3. Splitting into train/test sets
4. Building a feature preprocessor
5. Training, evaluating, and running inference with three models:
   - **KMeans** (unsupervised clustering)
   - **Decision Tree** (supervised classification)
   - **Logistic Regression** (supervised classification)

Each step mirrors what `src/main.py` does, broken out so you can inspect intermediate results.

## Imports

In [3]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
import sys

sys.path.insert(0, str(Path.cwd().parent))

from src.load_data import load_raw_data
from src.clean_data import clean_dataframe
from src.validate import validate_dataframe
from src.features import get_feature_preprocessor

# KMeans
from src.kmeans.kmeans_train import train_kmeans_model
from src.kmeans.kmeans_evaluate import evaluate_kmeans_model
from src.kmeans.kmeans_infer import run_kmeans_inference

# Logistic Regression
from src.logit_regression.logit_train import train_logit_model
from src.logit_regression.logit_evaluate import evaluate_logit_model
from src.logit_regression.logit_infer import run_logit_inference

# Decision Tree
from src.dtrees.dtrees_train import train_dtrees_model
from src.dtrees.dtrees_eval import evaluate_dtrees_model
from src.dtrees.dtrees_infer import run_dtrees_inference

## Configuration

The `SETTINGS` dictionary acts as the **single source of truth** for every configurable value in the pipeline. By centralising paths, column names, hyperparameters, and feature lists in one place we ensure that:

- No magic strings or numbers are scattered across the code.
- Changing a parameter (e.g. `test_size`, `random_state`, or adding a new feature column) only requires editing this one block.
- In a production setting this dictionary will be replaced by values loaded from `config.yml`, keeping the interface identical.

In [6]:
SETTINGS = {
    "raw_data_path":       Path("../data/raw/train.csv"),
    "processed_data_path": Path("../data/processed/clean.csv"),
    "kmeans_model_path":   Path("../models/kmeans_model.joblib"),
    "logit_model_path":    Path("../models/logit_model.joblib"),
    "dtrees_model_path":   Path("../models/dtrees_model.joblib"),
    "kmeans_predictions_path":  Path("../reports/kmeans_predictions.csv"),
    "logit_predictions_path":   Path("../reports/logit_predictions.csv"),
    "dtrees_predictions_path":  Path("../reports/dtrees_predictions.csv"),
    "target_column": "Heart Disease",
    "problem_type": "classification",
    "test_size":    0.2,
    "random_state": 42,
    "k_means_bins": 3,
    "max_iterations": 2000,
    "features": {
        "quantile_bin":        [],
        "categorical_onehot":  [],
        "numeric_passthrough": ["id", "Sex", "FBS over 120",
                                "EKG results", "Exercise angina",
                                "ST depression"],
        "ordinal_encode": ["Chest pain type", "Number of vessels fluro",
                           "Thallium", "Slope of ST"],
        "min_max_scaler": ["Age", "BP", "Cholesterol", "Max HR"],
        "n_bins": 3
    }
}

---
## Pipeline

The pipeline runs in sequential steps. Each step depends on the output of the previous one.

### Step 1 - Load Raw Data

Read the raw CSV from disk into a Pandas DataFrame. This is the untouched source data.

In [7]:
df_raw = load_raw_data(SETTINGS["raw_data_path"])
print(f"Loaded {len(df_raw)} rows, {len(df_raw.columns)} columns")
df_raw.head()

[load_data.load_raw_data] Loading raw data from: ..\data\raw\train.csv
[utils] Loading CSV from: ..\data\raw\train.csv
[utils]   Loaded 630000 rows x 15 columns.
Loaded 630000 rows, 15 columns


,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


### Step 2 - Clean Data

Apply cleaning rules (handle missing values, fix dtypes, etc.) to produce an analysis-ready DataFrame.

In [8]:
df_clean = clean_dataframe(df_raw, target_column=SETTINGS["target_column"])
print(f"Cleaned DataFrame: {df_clean.shape}")
df_clean.head()

[clean_data] Cleaning raw dataframe...
Cleaned DataFrame: (630000, 15)


,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


### Step 3 - Validate Cleaned Data

Run schema-level checks to make sure every expected column is present and has the right dtype before we proceed to modelling. This catches configuration drift early.

In [9]:
required_cols = (
    SETTINGS["features"]["quantile_bin"]
    + SETTINGS["features"]["categorical_onehot"]
    + SETTINGS["features"]["numeric_passthrough"]
    + SETTINGS["features"]["ordinal_encode"]
    + SETTINGS["features"]["min_max_scaler"]
    + [SETTINGS["target_column"]]
)
validate_dataframe(df_clean, required_columns=required_cols)
print("Validation passed.")

[validate] Running data validation checks...
[validate]   Shape: 630000 rows x 15 columns.
[validate]   All 15 required columns present.
[validate] Validation passed.
Validation passed.


### Step 4 - Train / Test Split

Split the data **before** fitting any transformers or models to prevent data leakage. For classification tasks we use stratified sampling so the class distribution is preserved in both sets.

In [10]:
X = df_clean.drop(columns=[SETTINGS["target_column"]])
y = df_clean[SETTINGS["target_column"]]

stratify_arg = None
if SETTINGS["problem_type"] == "classification":
    try:
        stratify_arg = y
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=SETTINGS["test_size"],
            random_state=SETTINGS["random_state"],
            stratify=stratify_arg,
        )
    except ValueError as e:
        print(f"Stratified split failed ({e}). Falling back to random split.")
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=SETTINGS["test_size"],
            random_state=SETTINGS["random_state"],
        )
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=SETTINGS["test_size"],
        random_state=SETTINGS["random_state"],
    )

print(f"Train size: {len(X_train)} rows | Test size: {len(X_test)} rows")

Train size: 504000 rows | Test size: 126000 rows


### Step 5 - Fail-Fast Feature Checks

Before building the preprocessor, verify that every column listed in `SETTINGS["features"]` actually exists in the training data. This prevents cryptic errors downstream.

In [11]:
all_feature_cols = (
    SETTINGS["features"]["quantile_bin"]
    + SETTINGS["features"]["categorical_onehot"]
    + SETTINGS["features"]["numeric_passthrough"]
    + SETTINGS["features"]["ordinal_encode"]
    + SETTINGS["features"]["min_max_scaler"]
)
missing_cols = [c for c in all_feature_cols if c not in X_train.columns]
if missing_cols:
    raise ValueError(
        f"The following configured feature columns are "
        f"missing from the data: {missing_cols}\n"
        "Update the 'features' section in SETTINGS to match "
        "your actual column names."
    )

for col in SETTINGS["features"]["quantile_bin"]:
    if not pd.api.types.is_numeric_dtype(X_train[col]):
        raise TypeError(
            f"Column '{col}' is listed under "
            "'quantile_bin' but is not numeric. "
            "Only numeric columns can be binned."
        )

print("All feature columns verified.")

All feature columns verified.


### Step 6 - Build Feature Preprocessor

Construct a `ColumnTransformer` that encodes the recipe for transforming raw features (scaling, encoding, binning). **No fitting happens here** - the preprocessor is passed into each model's training function where it gets fitted on the training data only.

In [12]:
preprocessor = get_feature_preprocessor(
    quantile_bin_cols=SETTINGS["features"]["quantile_bin"],
    categorical_onehot_cols=SETTINGS["features"]["categorical_onehot"],
    min_max_cols=SETTINGS["features"]["min_max_scaler"],
    ordinal_encode_cols=SETTINGS["features"]["ordinal_encode"],
    numeric_passthrough_cols=SETTINGS["features"]["numeric_passthrough"],
    n_bins=SETTINGS["features"]["n_bins"],
)
print("Preprocessor built.")
preprocessor

[features] Building feature preprocessor recipe...
Preprocessor built.


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('passthrough', ...), ('scaler', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``featur

---
## Step 7 - KMeans Clustering

KMeans is an **unsupervised** model - it groups rows into clusters without using the target label. This can reveal natural groupings in the patient data.

### 7a) Train KMeans

In [13]:
kmeans_model = train_kmeans_model(
    X_train=X_train,
    preprocessor=preprocessor,
    n_clusters=SETTINGS["k_means_bins"],
)
print("KMeans model trained.")

[kmeans.train] Training KMeans Pipeline
KMeans model trained.


### 7b) Evaluate KMeans

Since KMeans is unsupervised, evaluation uses an internal metric (e.g. silhouette score or inertia) rather than comparing against true labels.

In [14]:
kmeans_metric_value = evaluate_kmeans_model(model=kmeans_model, X_test=X_test)
print(f"KMeans evaluation metric: {kmeans_metric_value:.4f}")

[kmeans.evaluate] Evaluating silhouette score
KMeans evaluation metric: 0.5996


### 7c) KMeans Inference

Assign cluster labels to a small sample of unseen data.

In [15]:
X_example = X_test.head(5)
kmeans_predictions_df = run_kmeans_inference(model=kmeans_model, X_infer=X_example)
kmeans_predictions_df

[kmeans.infer] Predicting cluster_id


,prediction
307256,2
139643,0
148395,0
544541,1
619477,1


---
## Step 8 - Decision Tree

A **supervised** classifier that learns axis-aligned decision rules from the training data. Decision trees are highly interpretable - you can trace exactly why a patient was classified a certain way.

### 8a) Train Decision Tree

In [16]:
dtrees_model = train_dtrees_model(
    X_train=X_train,
    y_train=y_train,
    preprocessor=preprocessor,
    problem_type=SETTINGS["problem_type"],
)
print("Decision Tree model trained.")

[dtrees_train.train_model] Building Decision Tree pipeline...
[dtrees_train.train_model] Fitting on 504000 samples...
[dtrees_train.train_model] Training complete.
Decision Tree model trained.


### 8b) Evaluate Decision Tree

Measure performance on the held-out test set using the appropriate metric for the problem type (F1 weighted for classification, RMSE for regression).

In [17]:
dtrees_metrics = evaluate_dtrees_model(
    model=dtrees_model,
    X_test=X_test,
    y_test=y_test,
    prob_type=SETTINGS["problem_type"],
)
metric_label = "RMSE" if SETTINGS["problem_type"] == "regression" else "F1 (weighted)"
print(f"Decision Tree {metric_label}: {dtrees_metrics}")

[dtrees_eval.evaluate_model] Running predictions on test set...

[dtrees_eval.evaluate_model] Classification Report:
              precision    recall  f1-score   support

     Absence     0.8774    0.8563    0.8668     69509
    Presence     0.8283    0.8528    0.8404     56491

    accuracy                         0.8548    126000
   macro avg     0.8529    0.8546    0.8536    126000
weighted avg     0.8554    0.8548    0.8549    126000

[dtrees_eval.evaluate_model] Metrics Summary:
  TP                        48177
  TN                        59522
  FP                        9987
  FN                        8314
  Accuracy                  0.8547539682539682
  Precision                 0.8282958531050134
  Recall                    0.8528261138942487
  Specificity               0.8563207642175833
  F1-score                  0.8403820156120535
  False Positive Rate       0.14367923578241668

[dtrees_eval.evaluate_model] F1 Score: 0.8404
Decision Tree F1 (weighted): 0.840382015612053

### 8c) Decision Tree Inference

Generate predictions on the same example rows to compare across models.

In [18]:
dtrees_predictions_df = run_dtrees_inference(model=dtrees_model, X_infer=X_example)
dtrees_predictions_df

Running inference on 5 rows...
[dtrees_infer.run_inference] Inference complete.


,prediction
307256,Absence
139643,Presence
148395,Presence
544541,Presence
619477,Presence


---
## Step 9 - Logistic Regression

A **supervised** linear classifier that models the probability of heart disease. Logistic regression is a strong baseline for binary classification tasks and provides calibrated probability estimates.

### 9a) Train Logistic Regression

In [19]:
logit_model = train_logit_model(
    X_train, y_train, preprocessor,
    SETTINGS["problem_type"],
    SETTINGS["random_state"],
    SETTINGS["max_iterations"],
)
print("Logistic Regression model trained.")

[logit_train.logit_train]  Fitting model Pipeline
          (preprocess -> estimator)
Logistic Regression model trained.


c:\Users\vikyb\Desktop\IE-Folder\ml\MLOps\mlops-full\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### 9b) Evaluate Logistic Regression

Measure performance on the held-out test set.

In [20]:
logit_metrics = evaluate_logit_model(
    model=logit_model,
    X_test=X_test,
    y_test=y_test,
    prob_type=SETTINGS["problem_type"],
)
print(f"Logistic Regression {metric_label}: {logit_metrics}")

evaluate_model: evaluating model on held-out test set
Logistic Regression F1 (weighted): 0.8840315933461149


### 9c) Logistic Regression Inference

Generate predictions on the same example rows.

In [21]:
logit_predictions_df = run_logit_inference(model=logit_model, X_infer=X_example)
logit_predictions_df

run_inference: generating predictions


,prediction
307256,Absence
139643,Presence
148395,Presence
544541,Presence
619477,Presence


---
## Pipeline Complete

All three models have been trained, evaluated, and used for inference. In the production `src/main.py` script, the trained models and predictions are saved to disk automatically.